# Notebook 01 - Eksplorasi dan profiling data (Google Colab)

Notebook ini memakai Pandas pada sampel 100.000 baris untuk memahami schema dan kualitas data. Insight akhir tidak boleh disimpulkan hanya dari sampel awal ini.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyspark==3.5.5', 'pyarrow>=15,<21', 'python-dotenv>=1.0,<2.0',
    'psycopg2-binary>=2.9,<3.0', 'SQLAlchemy>=2.0,<3.0',
])

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Notebook ini khusus Google Colab. Gunakan versi notebooks/local untuk Jupyter.') from exc

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/USERNAME/big-data-retail-analytics.git'
PROJECT_ROOT = Path('/content/big-data-retail-analytics')
if not (PROJECT_ROOT / 'src').is_dir():
    if 'USERNAME' in REPOSITORY_URL:
        raise ValueError('Ganti USERNAME pada REPOSITORY_URL dengan username GitHub milikmu.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

COLAB_DATASET_ROOT = '/content/drive/MyDrive/Synthetic Retail Dataset - 10 Million Transactions'
DATASET_ROOT = Path(COLAB_DATASET_ROOT)
if not (DATASET_ROOT / 'retail_contaminated_dataset').is_dir():
    raise FileNotFoundError(f'Folder dataset tidak ditemukan: {DATASET_ROOT}')

OUTPUT_ROOT = Path('/content/drive/MyDrive/big-data-retail-analytics-output')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['DATASET_ROOT'] = str(DATASET_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT)
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g pyspark-shell'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import get_paths, validate_source_files

paths = get_paths('retail_contaminated_dataset')
validate_source_files(paths)
paths.source_dir

## 1. Membaca sampel transaksi dan tabel master

In [ ]:
SAMPLE_ROWS = 100_000
sales = pd.read_csv(
    paths.source_dir / 'sales_transactions.csv',
    nrows=SAMPLE_ROWS,
    parse_dates=['date'],
)
stores = pd.read_csv(paths.source_dir / 'store_master.csv', parse_dates=['opening_date'])
skus = pd.read_csv(paths.source_dir / 'sku_master.csv')
customers = pd.read_csv(
    paths.source_dir / 'customer_master.csv', parse_dates=['registration_date']
)
promotions = pd.read_csv(
    paths.source_dir / 'promotions.csv', parse_dates=['start_date', 'end_date']
)
inventory = pd.read_csv(
    paths.source_dir / 'inventory_snapshot.csv', parse_dates=['last_restock_date']
)
flags = pd.read_csv(
    paths.source_dir / 'sku_inventory_flags.csv',
    parse_dates=['window_start', 'window_end'],
)

print(f'Sampel transaksi: {len(sales):,} baris')
display(sales.head())

## 2. Ringkasan tabel dan schema

In [ ]:
tables = {
    'sales_sample': sales,
    'stores': stores,
    'skus': skus,
    'customers': customers,
    'promotions': promotions,
    'inventory': inventory,
    'flags': flags,
}
overview = pd.DataFrame([
    {
        'table': name,
        'rows': len(frame),
        'columns': frame.shape[1],
        'duplicate_rows': int(frame.duplicated().sum()),
        'missing_cells': int(frame.isna().sum().sum()),
    }
    for name, frame in tables.items()
])
display(overview)
display(pd.DataFrame({'dtype': sales.dtypes.astype(str), 'missing': sales.isna().sum()}))

## 3. Quality checks pada sampel

`promo_id` boleh kosong karena tidak semua transaksi memakai promosi.

In [ ]:
required = [
    'date', 'receipt_id', 'store_id', 'sku_id', 'customer_id',
    'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct'
]
calculated_total = (
    sales['quantity'] * sales['unit_price'] * (1 - sales['discount_pct'] / 100)
).round(2)
checks = {
    'sample_rows_100k': len(sales) == 100_000,
    'required_fields_complete': not sales[required].isna().any().any(),
    'quantity_positive': sales['quantity'].gt(0).all(),
    'price_nonnegative': sales['unit_price'].ge(0).all(),
    'discount_in_range': sales['discount_pct'].between(0, 100).all(),
    'total_value_formula': np.allclose(
        sales['total_value'], calculated_total, atol=0.02
    ),
    'store_keys_valid': sales['store_id'].isin(stores['store_id']).all(),
    'sku_keys_valid': sales['sku_id'].isin(skus['sku_id']).all(),
    'customer_keys_valid': sales['customer_id'].isin(customers['cust_id']).all(),
    'nonblank_promo_keys_valid': sales.loc[
        sales['promo_id'].notna(), 'promo_id'
    ].isin(promotions['promo_id']).all(),
}
check_table = pd.DataFrame({'check': checks.keys(), 'passed': checks.values()})
display(check_table)
assert check_table['passed'].all(), 'Ada quality check sampel yang gagal.'

## 4. Join dan kolom turunan

In [ ]:
store_dim = stores.rename(columns={'city': 'store_city'})
sku_dim = skus.rename(columns={'unit_price': 'list_unit_price'})
customer_dim = customers.rename(
    columns={'cust_id': 'customer_id', 'city': 'customer_city'}
)
eda = (
    sales.merge(store_dim, on='store_id', how='left', validate='many_to_one')
    .merge(sku_dim, on='sku_id', how='left', validate='many_to_one')
    .merge(customer_dim, on='customer_id', how='left', validate='many_to_one')
)
eda['month'] = eda['date'].dt.to_period('M').dt.to_timestamp()
eda['gross_sales'] = (eda['quantity'] * eda['unit_price']).round(2)
eda['discount_amount'] = (eda['gross_sales'] - eda['total_value']).round(2)
eda['gross_profit'] = (
    eda['total_value'] - eda['quantity'] * eda['cost_price']
).round(2)
display(eda.head())

## 5. Visualisasi eksplorasi sampel

Grafik berikut hanya untuk memahami struktur sampel. Grafik final dibuat dari agregasi data penuh pada Notebook 03.

In [ ]:
paths.charts_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
eda.groupby('category')['total_value'].sum().sort_values().plot.barh(ax=axes[0])
axes[0].set_title('Nilai penjualan per kategori - sampel')
axes[0].set_xlabel('Sales value')

eda.groupby('store_city')['total_value'].sum().nlargest(10).sort_values().plot.barh(ax=axes[1])
axes[1].set_title('10 kota teratas - sampel')
axes[1].set_xlabel('Sales value')

eda.groupby('channel')['total_value'].sum().sort_values(ascending=False).plot.bar(ax=axes[2])
axes[2].set_title('Nilai penjualan per kanal - sampel')
axes[2].set_ylabel('Sales value')
axes[2].tick_params(axis='x', rotation=0)

fig.tight_layout()
fig.savefig(paths.charts_dir / '00_sample_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

## Kesimpulan sementara

Tuliskan temuan mengenai schema, kualitas teknis, kolom yang perlu diturunkan, dan alasan data penuh diproses dengan PySpark. Jangan menuliskan insight bisnis final dari sampel awal.